# GenCast Mini — run on a Colab GPU kernel (via VS Code)

This notebook runs the same GenCast Mini pipeline already verified on the GPU
server, but against a Colab-hosted GPU kernel connected through VS Code's
Colab extension. It clones the repo fresh onto the Colab VM's own filesystem
(separate from this file's location on your laptop), downloads the public
GenCast Mini data, runs `gencast_mini_run.py` with the same GPU attention fix
already used before, and leaves `predictions.nc` plus the sanity/error-map/
RMSE diagnostics in `gencast_output/` on the Colab VM.

**Before running:** in VS Code, select the kernel for this notebook and pick
your connected Colab GPU runtime (not a local Python kernel). Then run the
cells below top to bottom, checking each one finishes before starting the
next -- some early steps (package installs, data download) take a few
minutes.


## 1. Check the GPU

In [ ]:
!nvidia-smi


## 2. Clone the repo onto the Colab VM

The Colab kernel runs on its own remote machine with its own filesystem -- it does not see files on your laptop. This clones a fresh copy there.

In [ ]:
REPO_DIR = "/content/weathernext"

import os
if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/swatibhalaik/weathernext.git {REPO_DIR}
else:
    print(f"{REPO_DIR} already exists, skipping clone")

%cd {REPO_DIR}


## 3. Install the model code and GPU-enabled JAX

In [ ]:
!pip install -q "git+https://github.com/google-deepmind/weathernext.git@08cf73625c9d12bd9aaa038868bcb2fe488f2a22"
!pip install -q -U "jax[cuda12]"


## 4. Install the remaining requirements

In [ ]:
!pip install -q -r requirements.txt


## 5. Sanity check: does JAX see the GPU?

This should print something like `[CudaDevice(id=0)]`. If it instead shows only a CPU device, stop here -- that means JAX installed without CUDA support and needs fixing before continuing.

In [ ]:
import jax
print(jax.devices())


## 6. Download the GenCast Mini data (~900 MB, public bucket, no auth needed)

In [ ]:
!mkdir -p gencast_data/dataset gencast_data/params gencast_data/stats

!wget -q -O gencast_data/dataset/source-era5_date-2019-03-29_res-1.0_levels-13_steps-30.nc \
  "https://storage.googleapis.com/dm_graphcast/gencast/dataset/source-era5_date-2019-03-29_res-1.0_levels-13_steps-30.nc"

!wget -q -O "gencast_data/params/GenCast_1p0deg_Mini_lt2019.npz" \
  "https://storage.googleapis.com/dm_graphcast/gencast/params/GenCast%201p0deg%20Mini%20%3C2019.npz"

!wget -q -O gencast_data/stats/diffs_stddev_by_level.nc \
  "https://storage.googleapis.com/dm_graphcast/gencast/stats/diffs_stddev_by_level.nc"
!wget -q -O gencast_data/stats/mean_by_level.nc \
  "https://storage.googleapis.com/dm_graphcast/gencast/stats/mean_by_level.nc"
!wget -q -O gencast_data/stats/stddev_by_level.nc \
  "https://storage.googleapis.com/dm_graphcast/gencast/stats/stddev_by_level.nc"
!wget -q -O gencast_data/stats/min_by_level.nc \
  "https://storage.googleapis.com/dm_graphcast/gencast/stats/min_by_level.nc"

!ls -la gencast_data/dataset gencast_data/params gencast_data/stats


Check the listing above shows real file sizes (hundreds of MB for the dataset/params files, not 0 bytes) before continuing.

## 7. Run the model

Same script, same `triblockdiag_mha` GPU attention fix already verified on
the H200 GPU server -- just a single T4 here, so the ensemble stays small.
If this fails with a CUDA out-of-memory error, lower
`--num-ensemble-members` to 2 rather than retrying blindly with the same
settings.

In [ ]:
!python gencast_mini_run.py --data-dir ./gencast_data --output-dir ./gencast_output --num-ensemble-members 4


## 8. Check the outputs

`predictions.nc` plus the sanity-check / error-map / RMSE-by-lead-time
diagnostic plots should now be in `gencast_output/` on the Colab VM.

In [ ]:
!ls -la gencast_output


## 9. (Optional) Download results back to your laptop

The files above live on the remote Colab VM, not your laptop. If your VS
Code Colab connection doesn't already sync `/content/weathernext` back to a
local folder, zip the output and download it manually (e.g. via the Colab
file browser, or `scp`/`rsync` if your connection method supports it).

In [ ]:
!zip -r gencast_output.zip gencast_output
print("Zipped -- download gencast_output.zip from the Colab VM's file browser if it isn't already synced to your laptop.")
